# AI Pays AI — Under a Cent

Call a priced capability with no payment: read the price and the 402, take a free call, then pay the same call from one prepaid key.

**~5 min · free · live network**

- Every call carries a price
- 402 lists the ways to pay
- One key pays from a balance

**Academy next:** [Agent Economy from Scratch](https://alexar76.github.io/aimarket-courses/agent-economy-course/)

_Tip: Runtime → Run all. Hub calls use the public AIMarket surface — no API key._


In [ ]:
# AIMarket School — AI Pays AI
# Cell 1: call a priced capability with no payment → HTTP 402 lists the ways to pay.
import urllib.request, urllib.error, json, uuid

HUB = 'https://modelmarket.dev'
CALL = {
    'product_id': 'prod-platon',
    'capability_id': 'platon.random@v1',
    'source_hub': 'https://oracles.modelmarket.dev/family',
    'input': {},
}

def invoke(headers=None):
    req = urllib.request.Request(HUB + '/ai-market/v2/invoke', data=json.dumps(CALL).encode(),
                                 headers={'content-type': 'application/json', **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read().decode() or '{}')

status, body = invoke()
print('status', status, '· price $%s' % body.get('price_usd'))
if status == 402:
    print('402 Payment Required — the hub names every way to pay:')
    for way in body.get('payment_ways', []):
        print('  ·', way['rail'], '→ header', way['header'], way.get('start', ''))
else:
    print('served without payment:', json.dumps(body)[:300])


In [ ]:
# Cell 2: the free trial — the same call, marked with a visitor id you choose.
wk = json.load(urllib.request.urlopen(HUB + '/.well-known/ai-market.json', timeout=20))
trial = wk.get('free_trial') or {}
print('free trial:', trial.get('max_invokes_per_visitor'), 'priced calls, renewed', trial.get('quota_window'))
visitor = {trial.get('visitor_header', 'X-AIMarket-Sandbox-Visitor'): 'school-' + uuid.uuid4().hex[:12]}
status, body = invoke(visitor)
print('status', status, '· sandbox', body.get('sandbox'))
if status == 200:
    print('output', json.dumps(body.get('output'))[:200])
    print('signed receipt fields:', sorted((body.get('receipt') or {}).keys()))
else:
    print(body.get('error'), body.get('detail', '')[:300])


In [ ]:
# Cell 3: one key pays every call from a prepaid balance — no wallet step per call.
# Your key never goes into the code or the saved output: set AIMARKET_API_KEY, or leave it
# unset and this cell opens one for you (a new key starts with the hub's signup credit).
import getpass, os
API_KEY = os.environ.get('AIMARKET_API_KEY', '')
credits = (wk.get('payment_rails') or {}).get('credits') or {}
if not API_KEY and credits.get('open_signup'):
    req = urllib.request.Request(HUB + '/ai-market/v2/accounts', data=json.dumps({'label': 'aimarket-school'}).encode(),
                                 headers={'content-type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=20) as r:
            opened = json.loads(r.read().decode())
        API_KEY = opened['api_key']
        os.environ['AIMARKET_API_KEY'] = API_KEY
        print('new key %s…%s, starting balance $%s — kept in this session only.' % (API_KEY[:9], API_KEY[-4:], opened.get('balance_usd')))
        print('To copy it for Claude Code, Cursor or VS Code run print(API_KEY) once, then clear that output.')
    except urllib.error.HTTPError as e:
        print('no key from this address right now:', e.code, e.read().decode()[:200])
if not API_KEY:
    API_KEY = getpass.getpass('Paste a key from ' + HUB + '/start (input hidden): ').strip()
if not API_KEY:
    print('Get a key at ' + HUB + '/start, top it up and run this cell again.')
else:
    status, body = invoke({'X-API-Key': API_KEY})
    if status == 200:
        print('status 200 · paid $%s from the balance' % body.get('price_usd'))
    elif status == 401:
        print('This hub does not know that key — copy it again from ' + HUB + '/start')
    else:
        print('status', status, body.get('error'), '— top the key up at ' + HUB + '/start')
    if status != 401:
        req = urllib.request.Request(HUB + '/ai-market/v2/account', headers={'X-API-Key': API_KEY})
        account = json.load(urllib.request.urlopen(req, timeout=20))
        print('balance $%s · spent $%s' % (account.get('balance_usd'), account.get('spent_usd')))


## Next

1. Open the Academy: [Agent Economy from Scratch](https://alexar76.github.io/aimarket-courses/agent-economy-course/)
2. Fill `# YOUR CODE HERE` stubs in `courselib/exercises.py`
3. `python labs/run_exercises.py --certificate "Your Name"`
4. Connect your own agent: `claude mcp add --transport http aimarket https://modelmarket.dev/mcp` — other clients and a prepaid key: https://modelmarket.dev/start
